# CD Atlas: UMAP Visualization

Generates UMAP plots for the full CD atlas and lineage-specific subclusters.

**Inputs** (exported from R subclustering scripts):
- `9p_integ_p19_upd_anno_umap.csv`    – full atlas UMAP (all cells)
- `myeloid/mye_iter4_umap.csv`        – myeloid subcluster UMAP
- `connective/con_iter2_umap.csv`     – fibroblast subcluster UMAP
- `t_cells/tcell_iter3_umap.csv`      – T cell subcluster UMAP
- `epithelial/epi_iter2_umap.csv`     – epithelial subcluster UMAP
- `cleaned_annoed_all_cell_types.h5ad` – annotated atlas (for NK cell IDs)

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cd/scrna/output"
MERGED_DIR = f"{DATA_DIR}/merged_cd"

In [ ]:
# Load UMAP coordinate files
umap_all = pd.read_csv(f"{MERGED_DIR}/9p_integ_p19_upd_anno_umap.csv", low_memory=False)
umap_mye = pd.read_csv(f"{DATA_DIR}/myeloid/mye_iter4_umap.csv")
umap_fib = pd.read_csv(f"{DATA_DIR}/connective/con_iter2_umap.csv")
umap_t   = pd.read_csv(f"{DATA_DIR}/t_cells/tcell_iter3_umap.csv")
umap_epi = pd.read_csv(f"{DATA_DIR}/epithelial/epi_iter2_umap.csv")

In [ ]:
# Reclassify NK cells in full atlas (from atlas h5ad obs)
atlas = sc.read_h5ad(f"{MERGED_DIR}/cleaned_annoed_all_cell_types.h5ad")
nk_ids = atlas[atlas.obs['cell category'] == 'NK'].obs_names.tolist()

umap_all['label_new'] = [
    'NK' if row['cell'] in nk_ids else row['primary_anno']
    for _, row in umap_all.iterrows()
]

## Color palettes

In [ ]:
# 15-color vibrant palette
my_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5"
]

# 15-color pastel palette
pastel_palette = [
    "#AEC6CF", "#FFB3C6", "#FF6666", "#FFCC99", "#99FF99",
    "#CCCCFF", "#b2e2eb", "#F0E68C", "#e4d4fa", "#FAD02E",
    "#D6AEDD", "#B3E5FC", "#FFDAB9", "#C1E1C1", "#F4C2C2"
]

## Plot helper functions

In [ ]:
def plot_umap(data, x_col, y_col, hue_col, alpha=0.7, palette=pastel_palette, font=11, size=5):
    """UMAP scatter with centroid labels (no legend)."""
    plt.figure(figsize=(6, 6))
    scatter = sns.scatterplot(
        data=data, x=x_col, y=y_col, hue=hue_col,
        s=size, linewidth=0, palette=palette, alpha=alpha
    )
    handles, labels = scatter.get_legend_handles_labels()
    scatter.legend([], [], frameon=False)
    texts = []
    for label in labels:
        cd = data[data[hue_col] == label]
        texts.append(plt.text(cd[x_col].mean(), cd[y_col].mean(),
                               label, fontsize=font, ha='center', va='center', color='black'))
    adjust_text(texts, arrowprops=dict(arrowstyle="->", color='black'))
    plt.grid(False); plt.xlabel('UMAP1'); plt.ylabel('UMAP2')
    plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()


def plot_umap_nolabel(data, x_col, y_col, hue_col, alpha=0.7, palette=my_palette, size=5):
    """UMAP scatter with alphabetically sorted colors, no labels."""
    plt.figure(figsize=(6, 6))
    sorted_labels  = sorted(data[hue_col].unique())
    color_mapping  = {lb: palette[i] for i, lb in enumerate(sorted_labels)}
    scatter = sns.scatterplot(
        data=data, x=x_col, y=y_col, hue=hue_col,
        s=size, linewidth=0, palette=color_mapping, alpha=alpha
    )
    scatter.legend([], [], frameon=False)
    plt.grid(False); plt.xlabel('UMAP1'); plt.ylabel('UMAP2')
    plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()

## Atlas-level UMAP

In [ ]:
# Full atlas by cell lineage (Fig. X)
plot_umap(umap_all, 'umap_1', 'umap_2', 'label_new', size=3, palette=my_palette)

In [ ]:
# Full atlas colored by dataset of origin (integration QC)
plot_umap_nolabel(umap_all, 'umap_1', 'umap_2', 'source', size=3, palette=my_palette)

## Lineage-specific UMAPs

In [ ]:
# Myeloid subcluster
plot_umap_nolabel(umap_mye, 'umapharmonymyeiter4_1', 'umapharmonymyeiter4_2',
                  'iter4_anno', size=3, palette=my_palette)

In [ ]:
# Fibroblast subcluster
plot_umap(umap_fib, 'umapharmonyconiter2_1', 'umapharmonyconiter2_2',
          'iter2_anno', size=3, palette=my_palette)

In [ ]:
# T cell subcluster
plot_umap_nolabel(umap_t, 'umapharmonytcelliter3_1', 'umapharmonytcelliter3_2',
                  'iter3_anno', size=3, palette=my_palette)

In [ ]:
# Epithelial subcluster
plot_umap_nolabel(umap_epi, 'umapharmonyepi1iter1_1', 'umapharmonyepi1iter1_2',
                  'iter1_anno', size=3, palette=my_palette)